# B0005 — round 2, corrected

Round 2 failed on every count: F0 scored 0.06896 where it should have reproduced round 1's 0.01053,
and the tuned result (0.01567) was worse than a constant-decrement baseline (0.01409). Four causes
were identified. All four are fixed here.

| # | What went wrong | Fix |
|---|---|---|
| 1 | **The model drifted between rounds.** Round 2 rebuilt the architecture instead of importing it: BatchNorm became unconditional, MaxPooling disappeared, and defaults changed (window 30 vs 10, kernels 5/9/13 vs 3/5/7, L2 added). So the round-2 baseline was broken before the search began. | Round 1's model restored exactly, with a **sanity gate** that stops the notebook if it does not reproduce 0.01053. |
| 2 | **The cycle index extrapolated.** `cycle/100` spans 0.01-1.33 in training and 1.34-1.68 at test. The two modes containing it (F1 0.222, F3 0.433) were the worst — the same unbounded-feature fault as the original MinMax scaler. | Cycle index **dropped**. Capacity already encodes position in life. |
| 3 | **Multivariate autoregression compounded error.** F2 (no cycle index) still scored 0.302: nine channels each feeding every other channel over 35 steps, with the noisier indicators corrupting the one channel we care about. | Indicators used through a **direct multi-horizon** output instead — the model maps the observed window straight to all 35 future values, so there is no feedback loop and nothing to compound. Indicators only ever need to be *observed*, never predicted. |
| 4 | **The search underfit.** Origins 89/94/99 train on less data than the test does, so the objective was harder than the target task; combined with a wide L2 range and a size penalty, the search selected heavy regularisation and a tiny model. Seed spread collapsed to 0.00007 — the model had become nearly deterministic, and it lost to a constant-decrement baseline. | Everything stays on **B0005 only**, since that is what Tian and Gao report. Instead: L2 capped at 1e-4 (it picked 0.0032), the size penalty halved, the weakest fold dropped, and — most importantly — **round 1's configuration is evaluated on the same validation as every trial, and is kept unless the search genuinely beats it there**. The search can no longer make the result worse. |

**Stage 0** sanity gate · **Stage B** recursive vs direct multi-horizon, with and without indicators
· **Stage A** constrained Bayesian search with a no-regression guard · **Stage C** variance check.

Run from the folder containing `B0005.mat`. Roughly 40-60 minutes.

In [1]:
import os, json, gc, time
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('op determinism unavailable:', e)
print('TF', tf.__version__, '| keras-tuner', kt.__version__)

TF 2.10.0 | keras-tuner 1.4.8


In [2]:
CELL        = 'B0005'          # B0005 only — this is what Tian and Gao report on
VAL_ORIGINS = [94, 99]         # forecast origins used for hyperparameter validation
VAL_HORIZON = 35               # same horizon as the test forecast
ROUND1_REF  = 0.01053           # round 1 untuned result the gate must reproduce
GATE_TOL    = 2.0               # stop if the gate is more than this factor off
SEEDS_GATE, SEEDS_SELECT, SEEDS_TRIAL, SEEDS_FINAL = 3, 5, 2, 10
MAX_TRIALS  = 30
PARAM_PENALTY = 1e-4           # halved: the previous value helped drive underfitting
EPOCHS, PATIENCE = 300, 30
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

from scipy.io import loadmat
IND_COLS = ['capacity','discharge_time','t_to_3p8V','t_to_3p5V','mean_voltage',
            'min_voltage','mean_temp','max_temp','temp_rise']

def load_cell(name):
    mat = loadmat(name + '.mat'); cycles = mat[name][0, 0]['cycle'][0]
    rows, n = [], 0
    for i in range(len(cycles)):
        r = cycles[i]
        if r['type'][0] != 'discharge': continue
        n += 1; d = r['data'][0][0]
        V = d['Voltage_measured'][0].astype(float); T = d['Temperature_measured'][0].astype(float)
        t = d['Time'][0].astype(float)
        tt = lambda vth: (float(t[np.where(V <= vth)[0][0]]) if len(np.where(V <= vth)[0]) else float(t[-1]))
        rows.append(dict(capacity=float(d['Capacity'][0][0]), discharge_time=float(t[-1]),
                         t_to_3p8V=tt(3.8), t_to_3p5V=tt(3.5), mean_voltage=float(V.mean()),
                         min_voltage=float(V.min()), mean_temp=float(T.mean()),
                         max_temp=float(T.max()), temp_rise=float(T.max()-T[0])))
    df = pd.DataFrame(rows); C0 = df['capacity'].iloc[0]
    S  = (df['capacity']/C0).values.astype(np.float32)
    start = int(0.8*len(df)) - 1                       # cycle 134 for a 168-cycle cell
    ref   = df[IND_COLS].iloc[0].replace(0, 1.0)
    Fall  = (df[IND_COLS]/ref).values.astype(np.float32)
    return dict(name=name, S=S, F_cap=Fall[:, :1].copy(), F_ind=Fall,
                n=len(df), C0=float(C0), start=start, horizon=len(df)-start)

CELLS = {CELL: load_cell(CELL)}
CC = CELLS[CELL]
CC['floor'] = float(np.sqrt(mean_squared_error(CC['S'][CC['start']:],
                            np.full(CC['horizon'], CC['S'][CC['start']-1]))))
print('%-6s %d cycles | forecast %d from index %d | SOH %.3f->%.3f | flat baseline %.5f'
      % (CC['name'], CC['n'], CC['horizon'], CC['start'], CC['S'][CC['start']],
         CC['S'][-1], CC['floor']))
for o in VAL_ORIGINS:
    print('  validation origin cycle %d -> forecasts %d cycles, ends at cycle %d (test starts at %d)'
          % (o, VAL_HORIZON, o+VAL_HORIZON-1, CC['start']+1))

B0005  168 cycles | forecast 35 from index 133 | SOH 0.747->0.714 | flat baseline 0.02927
  validation origin cycle 94 -> forecasts 35 cycles, ends at cycle 128 (test starts at 134)
  validation origin cycle 99 -> forecasts 35 cycles, ends at cycle 133 (test starts at 134)


In [3]:
def make_data(cell, mode, window, horizon, cut_idx=None):
    """mode: 'rec' one-step recursive (capacity only) | 'dir' direct multi-horizon.
       cut_idx: index the training data stops at (the forecast origin)."""
    F = cell['F_ind'] if mode == 'dir_ind' else cell['F_cap']
    S = cell['S']
    cut_idx = cell['start'] if cut_idx is None else cut_idx
    X, y = [], []
    if mode == 'rec':
        for e in range(window, cut_idx):
            X.append(F[e-window:e]); y.append([S[e]-S[e-1]])
    else:
        for e in range(window, cut_idx - horizon + 1):
            X.append(F[e-window:e]); y.append(S[e:e+horizon] - S[e-1])
    X, y = np.asarray(X, np.float32), np.asarray(y, np.float32)
    c = int(0.75*len(X))
    return X[:c], y[:c], X[c:], y[c:]

def build(window, n_feat, n_out, filters=32, ms_kernels=(3,5,7), lstm_units=32,
          n_cnn=1, n_dense=1, dense_units=64, dropout=0.1, l2=0.0,
          lr=1e-3, optimizer='adam', seed=SEED):
    """Round 1's architecture, unchanged: multi-scale CNN + BiLSTM, MaxPooling, no BatchNorm."""
    reg = regularizers.l2(l2) if l2 and l2 > 0 else None
    kw  = dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
               kernel_regularizer=reg)
    inp = Input(shape=(window, n_feat)); c = inp
    for _ in range(n_cnn):
        c = Concatenate()([Conv1D(filters, k, padding='same', activation='relu', **kw)(c)
                           for k in ms_kernels])
        c = Dropout(dropout, seed=seed)(c)
        if int(c.shape[1]) > 2:
            c = MaxPooling1D(2, padding='same')(c)
    c = GlobalAveragePooling1D()(c)
    r = Bidirectional(LSTM(lstm_units, **kw))(inp)
    h = Concatenate()([c, r])
    for _ in range(n_dense):
        h = Dense(dense_units, activation='relu', **kw)(h)
        h = Dropout(dropout, seed=seed)(h)
    out = Dense(n_out, activation='linear', **kw)(h)
    opt = {'adam': tf.keras.optimizers.Adam,
           'rmsprop': tf.keras.optimizers.RMSprop}[optimizer](lr)
    m = Model(inp, out); m.compile(optimizer=opt, loss='mse'); return m

def run_cell(cell, mode, kw, seed, batch=16, origin=None, horizon=None):
    """Train on data before `origin`, forecast `horizon` cycles from it, return RMSE on SOH.
       Defaults to the test setup: origin = the 80% point, horizon = the rest of the curve."""
    window = kw['window']
    st      = cell['start'] if origin is None else int(origin) - 1
    horizon = cell['horizon'] if horizon is None else int(horizon)
    horizon = min(horizon, len(cell['S']) - st)
    Xf, yf, Xv, yv = make_data(cell, mode, window, horizon, cut_idx=st)
    n_feat = (cell['F_ind'] if mode == 'dir_ind' else cell['F_cap']).shape[1]
    n_out  = 1 if mode == 'rec' else horizon
    K.clear_session(); gc.collect()
    np.random.seed(seed); tf.random.set_seed(seed)
    kk = {k: v for k, v in kw.items() if k != 'window'}
    m = build(window, n_feat, n_out, seed=seed, **kk)
    m.fit(Xf, yf, validation_data=(Xv, yv), epochs=EPOCHS, batch_size=batch, verbose=0,
          shuffle=True,
          callbacks=[EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True),
                     ReduceLROnPlateau('val_loss', factor=0.5, patience=10, min_lr=1e-7)])
    S = cell['S']
    F = cell['F_ind'] if mode == 'dir_ind' else cell['F_cap']
    if mode == 'rec':
        win, out, prev = list(F[st-window:st]), [], float(S[st-1])
        for _ in range(horizon):
            x = np.asarray(win[-window:], np.float32).reshape(1, window, n_feat)
            prev = float(np.clip(prev + float(m(x, training=False).numpy().ravel()[0]), 0.0, 1.2))
            out.append(prev); win.append(np.array([prev], np.float32))
        pred = np.asarray(out, np.float32)
    else:
        x = F[st-window:st].reshape(1, window, n_feat)
        pred = np.clip(S[st-1] + m(x, training=False).numpy().ravel(), 0.0, 1.2)
    return float(np.sqrt(mean_squared_error(S[st:st+horizon], pred))), m.count_params()

DEFAULTS = dict(window=10, filters=32, ms_kernels=(3,5,7), lstm_units=32,
                n_cnn=1, n_dense=1, dense_units=64, dropout=0.1, l2=0.0,
                lr=1e-3, optimizer='adam')          # round 1's configuration

def val_score(mode, kw, seed, batch=16):
    """Mean forecast RMSE across the validation origins — never touches the test region."""
    out, prm = [], 0
    for o in VAL_ORIGINS:
        v, prm = run_cell(CC, mode, kw, seed, batch=batch, origin=o, horizon=VAL_HORIZON)
        out.append(v)
    return float(np.mean(out)), prm

print('helpers ready')

helpers ready


## Stage 0 — sanity gate

The recursive capacity-only configuration is round 1's winner. If it does not reproduce 0.01053,
the harness has drifted again and nothing below it can be read.

In [4]:
g = [run_cell(CC, 'rec', DEFAULTS, SEED+s)[0] for s in range(SEEDS_GATE)]
gate = float(np.mean(g))
print('round 1 reference : %.5f' % ROUND1_REF)
print('gate reproduction : %.5f  (seeds %s)' % (gate, [round(v, 5) for v in g]))
if gate > ROUND1_REF*GATE_TOL:
    raise RuntimeError('SANITY GATE FAILED: %.5f is more than %.1fx the round 1 result %.5f. '
                       'The harness has drifted — fix before reading anything below.'
                       % (gate, GATE_TOL, ROUND1_REF))
print('gate passed — the model matches round 1')

round 1 reference : 0.01053
gate reproduction : 0.00913  (seeds [0.00701, 0.00907, 0.01132])
gate passed — the model matches round 1


## Stage B — recursive vs direct multi-horizon, with and without indicators

Selection is on forecasts from cycles 94 and 99, never on the test forecast from cycle 134.

In [5]:
MODES = [('rec',     'recursive, capacity only  (round 1)'),
         ('dir',     'direct multi-horizon, capacity only'),
         ('dir_ind', 'direct multi-horizon, + 8 health indicators')]
t0, sb = [], []
_t = time.time()
for mode, label in MODES:
    vs, ts, prm = [], [], None
    for s in range(SEEDS_SELECT):
        v, _ = val_score(mode, DEFAULTS, SEED+s); vs.append(v)
        r, prm = run_cell(CC, mode, DEFAULTS, SEED+s)
        ts.append(r)
    sb.append(dict(mode=mode, label=label, val_mean=float(np.mean(vs)),
                   val_std=float(np.std(vs, ddof=1)), test_mean=float(np.mean(ts)),
                   test_std=float(np.std(ts, ddof=1)), test_best=float(np.min(ts)),
                   params=int(prm)))
    print('%-9s %-44s val %.5f +/- %.5f | test %.5f +/- %.5f | best %.5f'
          % (mode, label, np.mean(vs), np.std(vs, ddof=1), np.mean(ts),
             np.std(ts, ddof=1), np.min(ts)))
SB = pd.DataFrame(sb).sort_values('val_mean').reset_index(drop=True)
BEST_MODE = SB.iloc[0]['mode']
print('\nselected on held-out-origin validation: %s' % BEST_MODE)
print('total %.0f s' % (time.time()-_t))

rec       recursive, capacity only  (round 1)          val 0.01305 +/- 0.00538 | test 0.01458 +/- 0.01259 | best 0.00668
dir       direct multi-horizon, capacity only          val 0.01747 +/- 0.00114 | test 0.01548 +/- 0.00165 | best 0.01359
dir_ind   direct multi-horizon, + 8 health indicators  val 0.01841 +/- 0.00202 | test 0.02559 +/- 0.00312 | best 0.02029

selected on held-out-origin validation: rec
total 497 s


## Stage A — constrained Bayesian search with a no-regression guard

Twelve hyperparameters, validated on forecasts from cycles 94 and 99. Round 1's configuration is
scored on **the same objective** as every trial, and is kept unless the search actually beats it
there — so this stage cannot produce a worse result than round 1.

In [6]:
OBJ = 'val_origins_penalised'

def hp_kwargs(hp):
    return dict(window      = hp.Choice('window', [5, 10, 20, 30]),
                n_cnn       = hp.Int('n_cnn', 1, 2),
                filters     = hp.Int('filters', 8, 64, step=8),
                ms_kernels  = tuple(int(v) for v in hp.Choice(
                                  'ms_kernels', ['3,5,7', '5,9,13', '3,7,11']).split(',')),
                lstm_units  = hp.Int('lstm_units', 8, 64, step=8),
                n_dense     = hp.Int('n_dense', 1, 2),
                dense_units = hp.Int('dense_units', 16, 128, step=16),
                dropout     = hp.Float('dropout', 0.0, 0.4, step=0.05),
                l2          = hp.Float('l2', 1e-7, 1e-4, sampling='log'),
                lr          = hp.Float('lr', 1e-4, 5e-3, sampling='log'),
                optimizer   = hp.Choice('optimizer', ['adam', 'rmsprop']))

def objective_of(kwargs, batch):
    sc, prm = [], 0
    for s in range(SEEDS_TRIAL):
        try:
            v, prm = val_score(BEST_MODE, kwargs, SEED+s, batch=batch)
            if not np.isfinite(v): v = 1e3
        except Exception as e:
            print('   fail (%s)' % type(e).__name__); v = 1e3
        sc.append(v)
    return float(np.mean(sc)), prm

class OriginBO(kt.BayesianOptimization):
    def run_trial(self, trial, *a, **kw):
        hp = trial.hyperparameters
        kwargs = hp_kwargs(hp); batch = hp.Choice('batch_size', [8, 16, 32])
        base, prm = objective_of(kwargs, batch)
        pen = PARAM_PENALTY*(prm/1e5)
        print('   trial %s: val %.5f + penalty %.5f = %.5f (%s params)'
              % (trial.trial_id, base, pen, base+pen, format(prm, ',')))
        return {OBJ: base + pen}

# round 1's configuration, scored on exactly the same objective as every trial
R1_OBJ, R1_PRM = objective_of(DEFAULTS, 16)
R1_OBJ += PARAM_PENALTY*(R1_PRM/1e5)
print('round 1 configuration on this objective: %.5f  (%s params)\n'
      % (R1_OBJ, format(R1_PRM, ',')))

_t = time.time()
tuner = OriginBO(hypermodel=lambda hp: build(hp_kwargs(hp)['window'], 1, 1),
                 objective=kt.Objective(OBJ, direction='min'),
                 max_trials=MAX_TRIALS, seed=SEED,
                 directory='bo_round2fix_b0005', project_name=BEST_MODE, overwrite=False)
tuner.search()
best_hp = tuner.get_best_hyperparameters(1)[0]
BO_OBJ  = min(t.score for t in tuner.oracle.trials.values()
              if t.score is not None and np.isfinite(t.score))
print('\nsearch done in %.0f s' % (time.time()-_t))
print('best trial objective %.5f  vs  round 1 configuration %.5f' % (BO_OBJ, R1_OBJ))

# no-regression guard: keep round 1 unless the search genuinely beat it on validation
USE_BO = BO_OBJ < R1_OBJ
if USE_BO:
    FINAL_KW, FINAL_BATCH, FINAL_SRC = hp_kwargs(best_hp), best_hp.values.get('batch_size', 16), 'Bayesian search'
    print('-> search improved on validation; using the tuned configuration')
    for k, v in sorted(best_hp.values.items()): print('   %-13s %s' % (k, v))
else:
    FINAL_KW, FINAL_BATCH, FINAL_SRC = DEFAULTS, 16, 'round 1 configuration (search did not beat it)'
    print('-> search did NOT beat round 1 on validation; keeping round 1 and reporting that')

Trial 30 Complete [00h 01m 29s]
val_origins_penalised: 0.03808149274402845

Best val_origins_penalised So Far: 0.006658967290600419
Total elapsed time: 01h 19m 50s

search done in 4790 s
best trial objective 0.00666  vs  round 1 configuration 0.00756
-> search improved on validation; using the tuned configuration
   batch_size    32
   dense_units   48
   dropout       0.35000000000000003
   filters       16
   l2            1.6960124954049813e-06
   lr            0.003250764901075103
   lstm_units    56
   ms_kernels    5,9,13
   n_cnn         1
   n_dense       2
   optimizer     rmsprop
   window        20


In [7]:
sc, prm = [], None
for s in range(SEEDS_FINAL):
    v, prm = run_cell(CC, BEST_MODE, FINAL_KW, SEED+1000+s, batch=FINAL_BATCH)
    sc.append(v)
TUNED = dict(mode=BEST_MODE, mean=float(np.mean(sc)), std=float(np.std(sc, ddof=1)),
             best=float(np.min(sc)), worst=float(np.max(sc)), params=int(prm),
             per_seed=[round(v, 5) for v in sc], hp=dict(best_hp.values),
             source=FINAL_SRC, used_bo=bool(USE_BO))
FLOOR = CC['floor']; CONST = 0.01409
untuned = float(SB[SB['mode'] == BEST_MODE]['test_mean'].iloc[0])
print('='*74)
print('round 1  untuned                        : %.5f' % ROUND1_REF)
print('round 2 (broken) tuned                  : 0.01567')
print('round 2 FIXED, %-8s untuned          : %.5f' % (BEST_MODE, untuned))
print('round 2 FIXED, %-8s final            : %.5f +/- %.5f  (best seed %.5f)'
      % (BEST_MODE, TUNED['mean'], TUNED['std'], TUNED['best']))
print('   configuration source                 : %s' % FINAL_SRC)
print('constant-decrement baseline             : %.5f' % CONST)
print('flat-line baseline                      : %.5f' % FLOOR)
print('params %s | skill vs flat %+.3f | beats const-decrement: %s'
      % (format(prm, ','), 1-TUNED['mean']/FLOOR, 'YES' if TUNED['mean'] < CONST else 'NO'))
print('='*74)

round 1  untuned                        : 0.01053
round 2 (broken) tuned                  : 0.01567
round 2 FIXED, rec      untuned          : 0.01458
round 2 FIXED, rec      final            : 0.01930 +/- 0.00087  (best seed 0.01816)
   configuration source                 : Bayesian search
constant-decrement baseline             : 0.01409
flat-line baseline                      : 0.02927
params 36,593 | skill vs flat +0.341 | beats const-decrement: NO


## Stage C — variance check and benchmark

In [8]:
PUB = [('CNN-BiLSTM-AM','Tian',0.00481),('CNN-BiLSTM','Tian',0.00595),('BiLSTM','Tian',0.00620),
       ('1D CNN-BiLSTM','Gao',0.00772),('CNN','Tian',0.00863),('1D CNN','Gao',0.00866),
       ('1D CNN-LSTM','Gao',0.00932),('1D CNN-RNN','Gao',0.01661),('RNN','Gao',0.01776),
       ('BiLSTM','Gao',0.01986),('LSTM','Gao',0.02302)]
lo, hi = min(v for _,_,v in PUB), max(v for _,_,v in PUB)
spread = TUNED['worst'] - TUNED['best']
print('published range across 11 models : %.5f .. %.5f (spread %.5f)' % (lo, hi, hi-lo))
print('our tuned model over 10 seeds    : %.5f .. %.5f (spread %.5f)'
      % (TUNED['best'], TUNED['worst'], spread))
print('per seed: %s' % TUNED['per_seed'])
print('VERDICT: seed spread of one architecture %s the published range'
      % ('EXCEEDS' if spread > (hi-lo) else 'is smaller than'))

rows  = [dict(method=m, source=s, rmse=v) for m, s, v in PUB]
rows += [dict(method='Ours round 1 (recursive, tuned)', source='this work', rmse=ROUND1_REF),
         dict(method='Ours round 2 fixed (%s, %s)' % (BEST_MODE, FINAL_SRC),
              source='this work', rmse=round(TUNED['mean'], 5)),
         dict(method='   our best seed', source='this work', rmse=round(TUNED['best'], 5)),
         dict(method='   constant-decrement baseline', source='reference', rmse=CONST),
         dict(method='   flat-line baseline', source='reference', rmse=round(FLOOR, 5))]
B = pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True); B.index = B.index+1
pd.set_option('display.width', 200); print('\n' + B.to_string())
B.to_csv('benchmark_b0005_round2_fixed.csv')
SB.to_csv('mode_ablation_b0005_round2_fixed.csv', index=False)
json.dump(dict(gate=gate, round1_ref=ROUND1_REF, floor=FLOOR, val_origins=VAL_ORIGINS,
               r1_objective=R1_OBJ, bo_objective=BO_OBJ, used_bo=bool(USE_BO),
               mode_ablation=sb, tuned=TUNED),
          open('round2_fixed_results_b0005.json','w'), indent=2, default=str)
print('\nsaved benchmark, mode ablation and results files')

published range across 11 models : 0.00481 .. 0.02302 (spread 0.01821)
our tuned model over 10 seeds    : 0.01816 .. 0.02081 (spread 0.00265)
per seed: [0.0186, 0.01818, 0.02081, 0.02009, 0.01998, 0.01987, 0.01816, 0.01916, 0.01901, 0.01918]
VERDICT: seed spread of one architecture is smaller than the published range

                                       method     source     rmse
1                               CNN-BiLSTM-AM       Tian  0.00481
2                                  CNN-BiLSTM       Tian  0.00595
3                                      BiLSTM       Tian  0.00620
4                               1D CNN-BiLSTM        Gao  0.00772
5                                         CNN       Tian  0.00863
6                                      1D CNN        Gao  0.00866
7                                 1D CNN-LSTM        Gao  0.00932
8             Ours round 1 (recursive, tuned)  this work  0.01053
9                 constant-decrement baseline  reference  0.01409
10                  